# DS201 · Lab 1: Mạng neural cơ bản (MLP) trên MNIST
## Bản thực hành tại lớp

Vừa nghe giảng vừa điền vào các chỗ `______`. Thứ tự các mục khớp với file đề `Lab 1.pdf`.

| Ký hiệu | Nghĩa |
|---|---|
| `______` kèm `# TODO n:` | Phần bạn tự viết. Chạy ô khi chưa điền sẽ báo `NameError: name '______' is not defined` |
| `assert ...` | Dòng tự kiểm tra. Không báo gì là đúng, báo `AssertionError` là còn sai |



## A.1. Kiểm tra môi trường

In [ ]:
import torch, torchvision, sklearn, matplotlib

print("torch       :", torch.__version__)
print("torchvision :", torchvision.__version__)
print("scikit-learn:", sklearn.__version__)
print("GPU         :", torch.cuda.is_available())

## A.2. Làm quen với tensor và nạp MNIST

In [ ]:
x = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])
print(x.shape, x.dtype)
print(x.view(3, 2))      # đổi shape
print(x @ x.T)           # [2,3] @ [3,2] -> [2,2]

In [ ]:
from torchvision import datasets, transforms

train_full = datasets.MNIST(root="data", train=True,  download=True,
                            transform=transforms.ToTensor())
test_ds    = datasets.MNIST(root="data", train=False, download=True,
                            transform=transforms.ToTensor())

print("Train:", train_full.data.shape, "| Test:", test_ds.data.shape)
print("Kiểu gốc:", train_full.data.dtype,
      "| pixel gốc:", train_full.data.min().item(),
      "->", train_full.data.max().item())

In [ ]:
img, label = train_full[0]
print("shape:", img.shape, "| min/max:", img.min().item(), img.max().item(),
      "| nhãn:", label)

**CỐ Ý GÂY LỖI:** vẽ thẳng tensor `[1, 28, 28]`.

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(img, cmap="gray")

In [ ]:
pairs = [(train_full, "train[0]"), (test_ds, "test[0]")]
fig, axes = plt.subplots(1, 2, figsize=(5, 2.6))
for ax, (ds, name) in zip(axes, pairs):
    img, label = ds[0]
    ax.imshow(img.squeeze(0), cmap="gray")   # [1,28,28] -> [28,28]
    ax.set_title(f"{name} · Nhãn: {label}")
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
train_counts = torch.bincount(train_full.targets, minlength=10)
test_counts  = torch.bincount(test_ds.targets,    minlength=10)

print("Chữ số | Train | Test")
for d in range(10):
    tr, te = train_counts[d].item(), test_counts[d].item()
    print(f"{d:>6} | {tr:>5} | {te:>4}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.bar(range(10), train_counts.numpy(), color="steelblue")
ax1.set_xticks(range(10))
ax1.set_xlabel("Chữ số"); ax1.set_ylabel("Số ảnh")
ax1.set_title("Phân bố nhãn tập train")

ax2.pie(train_counts.numpy(), labels=range(10), autopct="%.1f%%",
        startangle=90)
ax2.set_title("Tỉ lệ nhãn tập train")
plt.tight_layout(); plt.show()

## PHẦN B · Ô dùng chung (chạy trước mọi mục B.x)

Chuẩn hoá theo mean/std của MNIST, tách 54,000 train / 6,000 validation, batch 64, seed 42.

In [ ]:
import random, time
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

tf = transforms.Compose([
    transforms.ToTensor(),                        # [0,255] -> [0,1]
    transforms.Normalize((0.1307,), (0.3081,)),   # mean/std của MNIST
])
train_full = datasets.MNIST("data", train=True,  download=True,
                            transform=tf)
test_ds    = datasets.MNIST("data", train=False, download=True,
                            transform=tf)

g = torch.Generator().manual_seed(42)
train_ds, val_ds = random_split(train_full, [54000, 6000], generator=g)

BATCH = 64
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          generator=torch.Generator().manual_seed(42))
val_loader   = DataLoader(val_ds,  batch_size=256)
test_loader  = DataLoader(test_ds, batch_size=256)

print("device:", device)
print("train / val / test:", len(train_ds), len(val_ds), len(test_ds))

## B.1. Neuron, lớp và shape

In [ ]:
# TODO 1: một neuron 784 -> 1
neuron = ______
print("weight:", neuron.weight.shape, "| bias:", neuron.bias.shape)
print("Số tham số:", sum(p.numel() for p in neuron.parameters()))

assert sum(p.numel() for p in neuron.parameters()) == 785

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

set_seed(42)
# TODO 2: MLP 784 -> 128 -> 10, lớp cuối trả logits
mlp = ______
for name, p in mlp.named_parameters():
    print(f"{name:10s} {str(list(p.shape)):12s} {p.numel():>7,}")
print(f"{'Tổng':23s} {count_params(mlp):>7,}")

assert count_params(mlp) == 101_770, "Mô hình phải có 101,770 tham số"

In [ ]:
x, y = next(iter(train_loader))
print(f"{'input':10s}", list(x.shape))
h = x
for layer in mlp:
    h = ______           # TODO 3: cho h qua layer
    print(f"{layer.__class__.__name__:10s}", list(h.shape))

**CỐ Ý GÂY LỖI:** quên `nn.Flatten()`.

In [ ]:
no_flatten = nn.Sequential(nn.Linear(784, 128), nn.ReLU(),
                           nn.Linear(128, 10))
no_flatten(x)

## B.2. Vì sao cần hàm phi tuyến

In [ ]:
set_seed(0)
deep_linear = nn.Sequential(nn.Linear(4, 16), nn.Linear(16, 16),
                            nn.Linear(16, 2))
x4 = torch.randn(5, 4)

# gộp cả chuỗi thành một phép affine W x + b
L0, L1, L2 = deep_linear
# TODO 4: gộp 3 lớp thành một phép affine W x + b
W = ______
b = ______

print("Số tham số deep_linear:", count_params(deep_linear))
print("Shape W gộp:", list(W.shape),
      "-> tương đương nn.Linear(4, 2), chỉ 10 tham số")
diff = (deep_linear(x4) - (x4 @ W.T + b)).abs().max().item()
print("Sai khác lớn nhất:", diff)

assert diff < 1e-5, "Gộp chưa đúng: hai bên phải cho cùng kết quả"

In [ ]:
# vẫn L0, L1, L2, chỉ thêm ReLU
# TODO 5: L0, L1, L2 và ReLU xen giữa
deep_relu = ______
diff = (deep_relu(x4) - (x4 @ W.T + b)).abs().max().item()
print("Có ReLU, sai khác lớn nhất:", diff)

assert diff > 0.01, "Có ReLU thì hai bên phải khác nhau rõ rệt"

In [ ]:
z = torch.tensor([-2.0, -0.5, 0.0, 0.5, 2.0])
print("ReLU   :", torch.relu(z))
print("Sigmoid:", torch.sigmoid(z).round(decimals=3))
s = torch.softmax(z, dim=0)
print("Softmax:", s.round(decimals=3), "tổng =", s.sum().item())

## B.3. Logits, softmax, cross-entropy và autograd

In [ ]:
z = torch.tensor([[2.0, 1.0, 0.1]])     # logits, 1 mẫu 3 lớp
y = torch.tensor([0])                   # nhãn đúng: lớp 0

# TODO 6: logits -> xác suất
p = ______
print("p        :", p.round(decimals=3))
print("-log p_y :", round(-torch.log(p[0, 0]).item(), 3))
print("F.cross_entropy:", round(F.cross_entropy(z, y).item(), 3))

assert abs(p[0, 0].item() - 0.659) < 1e-3, "p chưa phải là softmax của z"

In [ ]:
z = torch.tensor([[2.0, 1.0, 0.1]], requires_grad=True)
loss = F.cross_entropy(z, y)
______                       # TODO 7: tính gradient

y_onehot = F.one_hot(y, num_classes=3).float()
p_minus_y = (torch.softmax(z, dim=1) - y_onehot).detach()
print("autograd dL/dz:", z.grad.round(decimals=3))
print("p - y         :", p_minus_y.round(decimals=3))

assert torch.allclose(z.grad, p_minus_y, atol=1e-6)

In [ ]:
t = torch.tensor(3.0, requires_grad=True)
f = t ** 2 + 2 * t          # df/dt = 2t + 2, tại t = 3 là 8
f.backward()
print(t.grad)

In [ ]:
z = torch.tensor([[2.0, 1.0, 0.1]], requires_grad=True)
bad = F.cross_entropy(torch.softmax(z, dim=1), y)    # softmax 2 lần
bad.backward()
print("loss đúng :", round(F.cross_entropy(z.detach(), y).item(), 3))
print("loss sai  :", round(bad.item(), 3))
print("grad sai  :", z.grad.round(decimals=3))

## B.4. Vòng lặp huấn luyện

In [ ]:
import math
print("Số batch mỗi epoch:", len(train_loader), "=",
      f"ceil(54000 / {BATCH}) =", math.ceil(54000 / BATCH))

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        ______                             # TODO 8a: xoá gradient cũ
        loss = ______                      # TODO 8b: forward + loss
        ______                             # TODO 8c: tính gradient
        ______                             # TODO 8d: cập nhật trọng số
        total += loss.item() * x.size(0); n += x.size(0)
    return total / n


@torch.no_grad()
def evaluate(model, loader, criterion):
    ______                             # TODO 9: chế độ đánh giá
    total, n, preds, trues = 0.0, 0, [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        total += criterion(logits, y).item() * x.size(0); n += x.size(0)
        preds.append(logits.argmax(dim=1).cpu()); trues.append(y.cpu())
    return total / n, torch.cat(trues).numpy(), torch.cat(preds).numpy()


def fit(model, epochs, lr=0.1, loader=None, verbose=True):
    loader = loader or train_loader
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = ______                 # TODO 10: SGD, lr=lr
    hist = {"train_loss": [], "val_loss": [], "val_acc": []}
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tr_loss = train_one_epoch(model, loader, criterion, optimizer)
        va_loss, yt, yp = evaluate(model, val_loader, criterion)
        va_acc = (yt == yp).mean()
        for k, v in zip(hist, (tr_loss, va_loss, va_acc)):
            hist[k].append(v)
        if verbose:
            print(f"epoch {ep:2d} | train_loss {tr_loss:.4f}"
                  f" | val_loss {va_loss:.4f} | val_acc {va_acc:.4f}"
                  f" | {time.time() - t0:.1f}s")
    return hist

**Kiểm tra nhanh trước khi huấn luyện thật.** Nếu ô dưới báo lỗi thì xem lại TODO 8 đến TODO 10, đừng chạy 20 epoch rồi mới phát hiện.

In [ ]:
from torch.utils.data import Subset

smoke = DataLoader(Subset(train_ds, range(2000)), batch_size=64)
set_seed(42)
m = nn.Sequential(nn.Flatten(), nn.Linear(784, 10))
crit = nn.CrossEntropyLoss()
opt = torch.optim.SGD(m.parameters(), lr=0.1)

l1 = train_one_epoch(m, smoke, crit, opt)
l2 = train_one_epoch(m, smoke, crit, opt)
print(f"loss lượt 1: {l1:.4f}  ->  lượt 2: {l2:.4f}")
assert l2 < l1, "Loss không giảm, xem lại 4 dòng ở TODO 8"

In [ ]:
set_seed(42)
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 128), nn.ReLU(),
    nn.Linear(128, 10),
)
history = fit(model, epochs=20, lr=0.1)

assert history["val_acc"][-1] > 0.95, "Sau 20 epoch val_acc phải trên 0.95"

**Lỗi im lặng, CỐ Ý:** bỏ `optimizer.zero_grad()`. Không có thông báo lỗi nào.

In [ ]:
def run_100_steps(zero_grad):
    set_seed(42)
    m = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(),
                      nn.Linear(128, 10)).to(device)
    opt = torch.optim.SGD(m.parameters(), lr=0.1)
    it = iter(train_loader)
    for step in range(1, 101):
        x, y = next(it)
        x, y = x.to(device), y.to(device)
        if zero_grad:
            opt.zero_grad()
        loss = F.cross_entropy(m(x), y)
        loss.backward()
        opt.step()
        if step % 20 == 0:
            print(f"  step {step:3d}  loss {loss.item():.4f}")

print("Có zero_grad():");   run_100_steps(True)
print("Thiếu zero_grad():"); run_100_steps(False)

## B.5. Chuẩn hoá, learning rate và khởi tạo

In [ ]:
from torch.utils.data import Subset

raw_tf = transforms.Compose([
    transforms.PILToTensor(),                  # uint8, vẫn 0..255
    transforms.Lambda(lambda t: t.float()),
])
raw_full = datasets.MNIST("data", train=True, download=True,
                          transform=raw_tf)
raw_loader = DataLoader(Subset(raw_full, train_ds.indices),
                        batch_size=BATCH, shuffle=True,
                        generator=torch.Generator().manual_seed(42))

x_raw, _ = next(iter(raw_loader))
print("pixel đưa vào mạng:", x_raw.min().item(), "->", x_raw.max().item())

def new_mlp():
    set_seed(42)
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(),
                         nn.Linear(128, 10))

print("KHÔNG chuẩn hoá:")
_ = fit(new_mlp(), epochs=1, lr=0.1, loader=raw_loader)
print("CÓ chuẩn hoá   :")
_ = fit(new_mlp(), epochs=1, lr=0.1)

In [ ]:
# TODO 11: bốn learning rate 1e-5, 0.01, 0.1, 1.0
for lr in ______:
    print(f"lr = {lr:g}")
    _ = fit(new_mlp(), epochs=2, lr=lr)

In [ ]:
zero_model = new_mlp()
for m in zero_model:
    if isinstance(m, nn.Linear):
        ______                       # TODO 12: weight và bias của m = 0

_ = fit(zero_model, epochs=1, lr=0.1)
W1 = zero_model[1].weight
diff = (W1 - W1[0]).abs().max().item()
print("128 hàng của W1 có khác nhau không?", diff)

assert diff == 0.0, "Khởi tạo bằng 0 thì 128 hàng phải giống hệt nhau"

## B.6. Đánh giá mô hình và đọc đường loss

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
best = ______   # TODO 13: epoch có val_loss nhỏ nhất, tính từ 1

plt.figure(figsize=(6, 3.6))
plt.plot(epochs, history["train_loss"], "o-", label="train loss")
plt.plot(epochs, history["val_loss"],   "s-", label="val loss")
plt.axvline(best, ls="--", c="gray",
            label=f"val loss thấp nhất (epoch {best})")
plt.xlabel("epoch"); plt.ylabel("cross-entropy")
plt.xticks(list(epochs)[::2])
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()
print("Epoch có val_loss thấp nhất:", best,
      "| val_loss =", round(min(history["val_loss"]), 4))

assert history["val_loss"][best - 1] == min(history["val_loss"])

In [ ]:
from sklearn.metrics import (accuracy_score, classification_report,
                             precision_recall_fscore_support,
                             confusion_matrix, ConfusionMatrixDisplay)

_, y_true, y_pred = evaluate(model, test_loader, nn.CrossEntropyLoss())
# TODO 14: precision, recall, F1 macro
p, r, f1, _ = ______

print(f"Accuracy       : {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision macro: {p:.4f}")
print(f"Recall macro   : {r:.4f}")
print(f"F1 macro       : {f1:.4f}")

In [ ]:
print(classification_report(y_true, y_pred, digits=4))

In [ ]:
# TODO 15: ma trận nhầm lẫn
cm = ______
disp = ConfusionMatrixDisplay(cm)
disp.plot(cmap="Blues", colorbar=False)
disp.ax_.set_xlabel("Nhãn dự đoán")
disp.ax_.set_ylabel("Nhãn thật")
plt.title("Ma trận nhầm lẫn trên tập test"); plt.tight_layout(); plt.show()

off = cm.copy(); np.fill_diagonal(off, 0)
print("5 cặp (thật -> đoán) nhầm nhiều nhất:")
for idx in off.ravel().argsort()[::-1][:5]:
    t, pr = divmod(idx, 10)
    print(f"  {t} -> {pr}: {off[t, pr]} ảnh")

assert cm.shape == (10, 10) and cm.sum() == 10_000